# MUTCD Multimodal RAG — General Notebook

This notebook keeps the original workflow in a clean order:

1. Set up Colab and the repository
2. Load provider keys
3. Check paths and restore the Qdrant database
4. Initialize the pipeline
5. Select one or more models
6. Test the system interactively
7. Load the fixed MUTCD-150 benchmark
8. Run a smoke test
9. Run the full benchmark
10. Check failures, selectively retry, and package outputs

The notebook is **not tied to Gemini, Claude, or Qwen**. Change the
selected aliases in the model-selection cell.

The figure-filter model is always set to `None`. Therefore, the
currently selected answer VLM also performs figure filtering.


## 0. Colab setup and repository update

Skip or modify this section when running outside Colab.


In [ ]:
import os
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
REPO_DIR = Path("/content/gemsrag") if IS_COLAB else Path.cwd()

if IS_COLAB:
    from google.colab import drive

    drive.mount(
        "/content/drive",
        force_remount=False,
    )

    os.environ["MRAG_ENV"] = "colab"

    if not REPO_DIR.exists():
        !git clone https://github.com/hannanazad/gemsrag.git /content/gemsrag
    else:
        %cd /content/gemsrag
        !git pull

    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    # Install the project environment.
    !pip install -q --index-url https://download.pytorch.org/whl/cu124                 torch==2.6.0 torchvision==0.21.0

    !pip install -q -r /content/gemsrag/requirements.txt

    !pip install -q --no-deps --force-reinstall                 "transformers>=4.49,<4.55"                 "huggingface_hub>=0.34,<0.35"                 "tokenizers>=0.21,<0.22"                 "torchao>=0.13,<0.14"

    !pip install -q -U openai anthropic

    print(
        "If this cell changed Torch or Transformers, restart the "
        "runtime once before continuing."
    )
else:
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    print("Local/HPRC repository:", REPO_DIR)


## 1. Load provider API keys

Keys only provide access. Model selection is done later.

In Colab, add whichever keys you need under the Secrets panel:

- `DASHSCOPE_API_KEY`
- `ANTHROPIC_API_KEY`
- `GEMINI_API_KEY`


In [ ]:
import os

if IS_COLAB:
    from google.colab import userdata

    def load_secret(
        environment_name: str,
        *secret_names: str,
    ) -> None:
        if os.environ.get(environment_name):
            print(environment_name, ": already loaded")
            return

        for secret_name in secret_names:
            try:
                value = userdata.get(secret_name)
            except Exception:
                value = None

            if value:
                os.environ[environment_name] = value
                print(
                    environment_name,
                    f": loaded from {secret_name!r}",
                )
                return

        print(environment_name, ": not loaded")

    load_secret(
        "DASHSCOPE_API_KEY",
        "DASHSCOPE_API_KEY",
        "VLM_API_KEY",
    )
    load_secret(
        "ANTHROPIC_API_KEY",
        "ANTHROPIC_API_KEY",
    )
    load_secret(
        "GEMINI_API_KEY",
        "GEMINI_API_KEY",
    )

if (
    os.environ.get("DASHSCOPE_API_KEY")
    and not os.environ.get("VLM_API_KEY")
):
    os.environ["VLM_API_KEY"] = os.environ[
        "DASHSCOPE_API_KEY"
    ]

print(
    {
        "DashScope": bool(
            os.environ.get("DASHSCOPE_API_KEY")
        ),
        "Anthropic": bool(
            os.environ.get("ANTHROPIC_API_KEY")
        ),
        "Gemini": bool(
            os.environ.get("GEMINI_API_KEY")
        ),
    }
)


## 2. Environment and path check


In [ ]:
import os
import sys

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from mrag.config import CFG

print("Environment :", CFG.environment)
print("Base dir    :", CFG.base_dir)
print("PDF path    :", CFG.pdf_path)
print("Qdrant dir  :", CFG.qdrant_dir)
print("Cache dir   :", CFG.cache_dir)
print("HF cache    :", CFG.hf_home)

assert CFG.pdf_path.exists(), (
    f"No MUTCD PDF was found at {CFG.pdf_path}. "
    f"Place it under {CFG.base_dir}."
)


## 3. Restore or build the Qdrant vector store

If a valid local store exists, it is reused. Otherwise, the Drive
snapshot is restored. If neither exists, ingestion runs once.


In [ ]:
import json
import shutil
import tarfile
from pathlib import Path

from qdrant_client import QdrantClient

local_qdrant = Path(CFG.qdrant_dir)
drive_qdrant_tar = (
    Path(CFG.base_dir)
    / "qdrant_db.tar"
)

def qdrant_is_ready(
    path: Path,
) -> bool:
    if not path.exists():
        return False

    try:
        client = QdrantClient(
            path=str(path)
        )
        try:
            names = {
                collection.name
                for collection
                in client.get_collections().collections
            }
            return CFG.coll_chunks in names
        finally:
            client.close()
    except Exception:
        return False

def figure_cache_is_current() -> bool:
    try:
        with Path(CFG.figures_jsonl).open(
            "r",
            encoding="utf-8",
        ) as stream:
            first = json.loads(next(stream))

        return str(
            first.get(
                "extraction_method",
                "",
            )
        ).startswith("caption_below_v2")
    except Exception:
        return False

if (
    qdrant_is_ready(local_qdrant)
    and figure_cache_is_current()
):
    print(
        "Using existing local Qdrant:",
        local_qdrant,
    )

elif (
    drive_qdrant_tar.exists()
    and figure_cache_is_current()
):
    print(
        "Restoring Qdrant snapshot:",
        drive_qdrant_tar,
    )

    shutil.rmtree(
        local_qdrant,
        ignore_errors=True,
    )
    local_qdrant.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with tarfile.open(
        drive_qdrant_tar,
        "r",
    ) as archive:
        archive.extractall(
            local_qdrant.parent
        )

    assert qdrant_is_ready(
        local_qdrant
    )

    print("Qdrant restored.")

else:
    print(
        "No current Qdrant store was found. "
        "Running ingestion."
    )

    if IS_COLAB:
        %cd /content/gemsrag
        !python scripts/ingest_v4.py
    else:
        import subprocess
        import sys

        subprocess.run(
            [
                sys.executable,
                str(
                    REPO_DIR
                    / "scripts"
                    / "ingest_v4.py"
                ),
            ],
            cwd=str(REPO_DIR),
            check=True,
        )

    assert qdrant_is_ready(
        local_qdrant
    )
    assert figure_cache_is_current()

    print("Ingestion completed.")


### Optional: save the local Qdrant store back to Drive

Run this after a new ingestion. It is safe to skip when a current
snapshot already exists.


In [ ]:
SAVE_QDRANT_SNAPSHOT = False

if SAVE_QDRANT_SNAPSHOT:
    if CFG.environment != "colab":
        raise RuntimeError(
            "The standard snapshot location is configured for Colab."
        )

    if not local_qdrant.exists():
        raise FileNotFoundError(
            local_qdrant
        )

    drive_qdrant_tar.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "Saving:",
        local_qdrant,
        "->",
        drive_qdrant_tar,
    )

    with tarfile.open(
        drive_qdrant_tar,
        "w",
    ) as archive:
        archive.add(
            local_qdrant,
            arcname=local_qdrant.name,
        )

    print("Snapshot saved.")
else:
    print("Snapshot cell is disabled.")


## 4. Initialize the pipeline


In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(name)s - %(message)s",
)

from mrag.ask import init_pipeline, ask

pipeline = init_pipeline()

print(
    "VLM:",
    pipeline.vlm.loaded_name
    if pipeline.vlm
    else "not loaded",
)
print(
    "KG:",
    pipeline.kg.g.number_of_nodes(),
    "nodes,",
    pipeline.kg.g.number_of_edges(),
    "edges",
)


## 5. Select model or models

Edit `SELECTED_MODELS` only.

Use aliases from `CFG.list_vlm_models()` or from the benchmark model
registry.

Examples:

```python
SELECTED_MODELS = {"frontier_qwen"}
SELECTED_MODELS = {"balanced_claude"}
SELECTED_MODELS = {"frontier_gemini", "balanced_gemini", "fast_gemini"}
```

`CFG.set_filter_model(None)` is fixed below. The selected VLM performs
both figure filtering and final answering.


In [ ]:
from mrag.config import (
    CFG,
    VLM_API_MODELS,
    provider_of_model,
)

# ---------------------------------------------------------------------------
# Select model or models
# Use a list so the execution order stays predictable.
# ---------------------------------------------------------------------------

SELECTED_MODELS = [
    "balanced_gemini",
]

# Other valid examples:
#
# SELECTED_MODELS = [
#     "frontier_gemini",
# ]
#
# SELECTED_MODELS = [
#     "balanced_gemini",
# ]
#
# SELECTED_MODELS = [
#     "frontier_gemini",
#     "balanced_gemini",
#     "fast_gemini",
# ]

PROMPT_STYLE = "fewshot"

# The selected VLM performs both figure filtering and final answering.
CFG.use_vlm_figure_filter = True
CFG.set_filter_model(None)

MODEL_REGISTRY = []

for model_alias in SELECTED_MODELS:
    if model_alias not in VLM_API_MODELS:
        raise ValueError(
            f"Unknown model alias: {model_alias!r}. "
            "Check CFG.list_vlm_models()."
        )

    resolved_model_id = VLM_API_MODELS[model_alias]
    provider = provider_of_model(resolved_model_id)

    MODEL_REGISTRY.append(
        {
            "alias": model_alias,
            "selector": model_alias,
            "provider": provider,
            "enabled": True,
        }
    )

    print(
        f"{model_alias} -> {resolved_model_id} "
        f"[provider={provider}]"
    )

# Leave the first selected model active for interactive use.
CFG.set_vlm_model(SELECTED_MODELS[0])
CFG.set_filter_model(None)

assert MODEL_REGISTRY, (
    "MODEL_REGISTRY is empty. Check SELECTED_MODELS."
)

print("\nModels supplied to run_benchmark:")
print(MODEL_REGISTRY)

print(
    "\nEffective figure-filter model:",
    CFG.vlm_model_filter or CFG.vlm_model_api,
)

## 6. Optional interactive questions

Change `RUN_INTERACTIVE_TESTS` to `True` when needed.


In [ ]:
RUN_INTERACTIVE_TESTS = True

if RUN_INTERACTIVE_TESTS:
    interactive_model = next(
        iter(SELECTED_MODELS)
    )

    CFG.set_vlm_model(
        interactive_model
    )
    CFG.set_filter_model(None)

    _ = ask(
        "Explain Figure 2B-1 and the plaques it shows.",
        show_scores=True,
    )

    _ = ask(
        "What does MUTCD say about pedestrian hybrid beacons?",
        show_text=True,
    )

    _ = ask(
        "What does 'shall' mean in the MUTCD?",
        show_scores=True,
    )
else:
    print("Interactive tests are disabled.")


## 7. Optional knowledge-graph and retrieval inspection


In [ ]:
RUN_DEBUG_INSPECTION = False

if RUN_DEBUG_INSPECTION:
    graph = pipeline.kg.g

    print(
        graph.number_of_nodes(),
        "nodes,",
        graph.number_of_edges(),
        "edges",
    )

    node_id = "signcode:R1-1"

    if node_id in graph:
        print(
            "Node:",
            graph.nodes[node_id],
        )
        print(
            "Neighbors:",
            list(
                graph.neighbors(
                    node_id
                )
            )[:30],
        )

    retrieval_result = (
        pipeline.retriever.retrieve(
            "Explain Figure 2B-1 and the plaques it shows."
        )
    )

    print(
        "Router:",
        retrieval_result.debug.get(
            "figure_router"
        ),
    )
    print(
        "Chunks:",
        len(
            retrieval_result.chunks
        ),
    )
    print(
        "Figures:",
        len(
            retrieval_result.figures
        ),
    )
    print(
        "Pages:",
        len(
            retrieval_result.pages
        ),
    )
else:
    print("Debug inspection is disabled.")


## 8. Load the MUTCD-150 benchmark

Runtime files come from the GitHub repository. Outputs are written to
Google Drive.

The immutable question-only SHA-256 is verified before a run begins.


In [ ]:
import hashlib
import importlib
import json
import shutil
import sys
from pathlib import Path

REPO_BENCH_DIR = (
    REPO_DIR
    / "benchmarks"
    / "mutcd150"
    / "v1"
)

DRIVE_BENCH_DIR = (
    Path(CFG.base_dir)
    / "benchmarks"
    / "mutcd150"
    / "v1"
)
DRIVE_BENCH_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RUNNER_PATH = (
    REPO_BENCH_DIR
    / "mutcd_benchmark_runner.py"
)
QUESTIONS_PATH = (
    REPO_BENCH_DIR
    / "mutcd_benchmark_questions_v1.jsonl"
)
REPO_MODEL_REGISTRY_PATH = (
    REPO_BENCH_DIR
    / "model_registry.json"
)
DRIVE_MODEL_REGISTRY_PATH = (
    DRIVE_BENCH_DIR
    / "model_registry.json"
)
RUNTIME_MANIFEST_PATH = (
    REPO_BENCH_DIR
    / "runtime_manifest.json"
)

EXPECTED_QUESTIONS_SHA256 = (
    "3a04b1d620a80704eefac34c565449a0cb8814e781dd6d73b8"
    "afb77318b954b2"
)

def sha256_file(
    path: Path,
) -> str:
    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as stream:
        for block in iter(
            lambda: stream.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(
                block
            )

    return digest.hexdigest()

for required_path in [
    RUNNER_PATH,
    QUESTIONS_PATH,
    REPO_MODEL_REGISTRY_PATH,
    RUNTIME_MANIFEST_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            required_path
        )

actual_hash = sha256_file(
    QUESTIONS_PATH
)

if (
    actual_hash
    != EXPECTED_QUESTIONS_SHA256
):
    raise ValueError(
        "The benchmark question file was modified.\n"
        f"Expected: {EXPECTED_QUESTIONS_SHA256}\n"
        f"Received: {actual_hash}"
    )

if not DRIVE_MODEL_REGISTRY_PATH.exists():
    shutil.copy2(
        REPO_MODEL_REGISTRY_PATH,
        DRIVE_MODEL_REGISTRY_PATH,
    )

MODEL_REGISTRY_PATH = (
    DRIVE_MODEL_REGISTRY_PATH
)

if str(REPO_BENCH_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(REPO_BENCH_DIR),
    )

if "mutcd_benchmark_runner" in sys.modules:
    del sys.modules[
        "mutcd_benchmark_runner"
    ]

runner = importlib.import_module(
    "mutcd_benchmark_runner"
)

run_benchmark = runner.run_benchmark
validate_run_outputs = (
    runner.validate_run_outputs
)

print("Runner version:", runner.RUNNER_VERSION)
print("Question hash :", actual_hash)
print("Registry      :", MODEL_REGISTRY_PATH)

OUTPUT_ROOT = (
    Path(CFG.base_dir)
    / "benchmark_runs"
    / "mutcd150_v1"
)
OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


## 9. Build the selected model list

This does not permanently rewrite the model registry. It only creates
the list used in the current notebook session.


In [ ]:
import json

registry_document = json.loads(
    MODEL_REGISTRY_PATH.read_text(
        encoding="utf-8"
    )
)

all_model_specs = registry_document["models"]

MODEL_REGISTRY = [
    model
    for model in all_model_specs
    if (
        model.get("alias")
        in SELECTED_MODELS
        or model.get("selector")
        in SELECTED_MODELS
    )
]

if not MODEL_REGISTRY:
    raise ValueError(
        "None of SELECTED_MODELS were found in model_registry.json."
    )

print("Models for this session:")

for model in MODEL_REGISTRY:
    print(
        " ",
        model["alias"],
        "->",
        model["selector"],
        "[",
        model.get("provider"),
        "]",
    )

# Keep the figure filter following whichever answer model the runner
# selects during the loop.
CFG.use_vlm_figure_filter = True
CFG.set_filter_model(None)


## 10. Five-question smoke test

Run this first. It covers text, table, figure, and mixed questions.

Change `RUN_SMOKE_TEST` to `True` when ready.


In [ ]:
import json

RUN_SMOKE_TEST = True

# Put the model aliases directly here.
# No model_registry.json filtering is involved.
SMOKE_MODELS = [
    "balanced_gemini",
]

SMOKE_QUESTION_IDS = [
    "T017",
    "TB005",
    "F005",
    "M008",
    "M024",
]

# Use a new run ID.
SMOKE_RUN_ID = "mutcd150_smoke_balanced_gemini_01"

if RUN_SMOKE_TEST:
    assert SMOKE_MODELS, "SMOKE_MODELS is empty."

    # Confirm the model and provider before running.
    for model_alias in SMOKE_MODELS:
        resolved_model = CFG.set_vlm_model(model_alias)

        print(
            f"{model_alias} -> {resolved_model}"
        )

    # The selected answer model also performs figure filtering.
    CFG.use_vlm_figure_filter = True
    CFG.set_filter_model(None)

    print("\nModels passed to runner:", SMOKE_MODELS)
    print(
        "Figure-filter override:",
        CFG.vlm_model_filter,
    )
    print(
        "Current effective filter model:",
        CFG.vlm_model_api,
    )

    smoke_paths = run_benchmark(
        CFG=CFG,
        ask_fn=ask,
        questions_path=QUESTIONS_PATH,
        output_root=OUTPUT_ROOT,
        run_id=SMOKE_RUN_ID,

        # IMPORTANT: pass aliases directly.
        models=SMOKE_MODELS,

        prompt_style=PROMPT_STYLE,
        replicates=1,
        question_ids=SMOKE_QUESTION_IDS,
        show_scores=True,
        show_text=True,
        max_attempts=5,
        retry_base_seconds=8.0,
        inter_request_seconds=2.0,
        seed=20260710,
        resume=True,
        rerun_errors=True,
        strict_benchmark_hash=True,
        strict_question_count=True,
        store_serialized_return=True,
        store_raw_debug=True,
        echo_answer_preview=True,
    )

    smoke_summary = validate_run_outputs(
        smoke_paths["run_dir"],
        SMOKE_RUN_ID,
    )

    print(
        json.dumps(
            smoke_summary,
            indent=2,
        )
    )

    expected_records = (
        len(SMOKE_QUESTION_IDS)
        * len(SMOKE_MODELS)
    )

    assert smoke_summary["ok_records"] == expected_records, (
        f"Expected {expected_records} successful records, "
        f"but received {smoke_summary['ok_records']}."
    )

    assert smoke_summary["error_records"] == 0
    assert smoke_summary["unresolved_error_records"] == 0

    print(
        f"\nSmoke test completed: "
        f"{smoke_summary['ok_records']}/{expected_records}"
    )

else:
    print("Smoke test is disabled.")

## 11. Full 150-question benchmark

Keep `RUN_FULL_BENCHMARK=False` until the smoke-test output is clean.

Change `PRIMARY_RUN_ID` whenever the selected models or configuration
changes.


In [ ]:
import json

# ---------------------------------------------------------------------------
# Full MUTCD-150 benchmark
# ---------------------------------------------------------------------------

RUN_FULL_BENCHMARK = True

# Put the model alias directly here.
# Run one model at a time to keep outputs and retries separate.
BENCHMARK_MODELS = [
    "balanced_gemini",
]

# Use a unique run ID for this model and configuration.
PRIMARY_RUN_ID = "mutcd150_primary_balanced_gemini_01"

if RUN_FULL_BENCHMARK:
    if not BENCHMARK_MODELS:
        raise ValueError(
            "BENCHMARK_MODELS is empty. Add at least one model alias."
        )

    # Verify every selected model before beginning the 150-question run.
    print("Models selected for the full run:")

    for model_alias in BENCHMARK_MODELS:
        resolved_model = CFG.set_vlm_model(model_alias)

        print(
            f"  {model_alias} -> {resolved_model}"
        )

    # The currently selected VLM performs both:
    # 1. figure filtering
    # 2. final answer generation
    CFG.use_vlm_figure_filter = True
    CFG.set_filter_model(None)

    print(
        "\nFigure-filter override:",
        CFG.vlm_model_filter,
    )
    print(
        "Filter policy:",
        "follow each selected answer model",
    )
    print(
        "Run ID:",
        PRIMARY_RUN_ID,
    )

    primary_paths = run_benchmark(
        CFG=CFG,
        ask_fn=ask,
        questions_path=QUESTIONS_PATH,
        output_root=OUTPUT_ROOT,
        run_id=PRIMARY_RUN_ID,

        # Pass model aliases directly.
        models=BENCHMARK_MODELS,

        prompt_style=PROMPT_STYLE,
        replicates=1,

        # Run all 150 questions.
        max_questions=None,
        question_ids=None,

        show_scores=False,
        show_text=False,
        max_attempts=3,
        retry_base_seconds=8.0,
        inter_request_seconds=2.0,
        seed=20260710,
        resume=True,
        rerun_errors=True,
        strict_benchmark_hash=True,
        strict_question_count=True,
        store_serialized_return=True,
        store_raw_debug=True,
        echo_answer_preview=True,
    )

    primary_summary = validate_run_outputs(
        primary_paths["run_dir"],
        PRIMARY_RUN_ID,
    )

    print(
        json.dumps(
            primary_summary,
            indent=2,
        )
    )

    expected_records = (
        150 * len(BENCHMARK_MODELS)
    )

    assert primary_summary["ok_records"] == expected_records, (
        f"Expected {expected_records} successful records, "
        f"but received {primary_summary['ok_records']}. "
        "Inspect the errors file and run validation before retrying."
    )

    assert primary_summary["error_records"] == 0, (
        "The run contains unresolved error records."
    )

    assert (
        primary_summary["unresolved_error_records"]
        == 0
    ), (
        "The run contains unresolved provider or runtime failures."
    )

    print(
        "\nFull benchmark completed successfully:",
        f"{primary_summary['ok_records']}/{expected_records}",
    )
    print(
        "Run directory:",
        primary_paths["run_dir"],
    )

else:
    print(
        "Full benchmark is disabled. "
        "Set RUN_FULL_BENCHMARK=True after the smoke test passes."
    )

## 12. Check a completed run for runtime problems

Enter the run ID you want to inspect. This checks missing, empty,
error, retry, and provider-truncation records. It does not judge
substantive answer correctness.


In [ ]:
import json
import re
from pathlib import Path

RUN_ID_TO_CHECK = PRIMARY_RUN_ID

def read_jsonl(
    path: Path,
) -> list[dict]:
    if not path.exists():
        return []

    return [
        json.loads(line)
        for line in path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]

def natural_sort(
    question_id: str,
):
    match = re.match(
        r"([A-Za-z]+)(\d+)",
        question_id,
    )

    if not match:
        return (
            question_id,
            0,
        )

    return (
        match.group(1),
        int(match.group(2)),
    )

run_dir = (
    OUTPUT_ROOT
    / RUN_ID_TO_CHECK
)

answers = read_jsonl(
    run_dir
    / f"answers_{RUN_ID_TO_CHECK}.jsonl"
)
errors = read_jsonl(
    run_dir
    / f"errors_{RUN_ID_TO_CHECK}.jsonl"
)

retried_ids = set()
invalid_ids = set()
truncated_ids = set()

for record in answers:
    question_id = str(
        record.get(
            "question_id",
            "",
        )
    )

    if int(
        record.get(
            "attempts",
            1,
        )
        or 1
    ) > 1:
        retried_ids.add(
            question_id
        )

    answer = str(
        record.get(
            "answer",
            "",
        )
        or ""
    ).strip()

    serialized = (
        record.get(
            "serialized_return"
        )
        or {}
    )
    serialized_answer = str(
        serialized.get(
            "answer",
            "",
        )
        or ""
    ).strip()

    if (
        str(
            record.get(
                "status",
                "",
            )
        ).lower()
        != "ok"
        or not answer
        or not serialized_answer
        or answer.startswith(
            "### "
        )
        or answer.lower().startswith(
            (
                "(vlm error:",
                "vlm error:",
            )
        )
    ):
        invalid_ids.add(
            question_id
        )

    stop_reason = (
        serialized.get(
            "debug",
            {}
        )
        .get(
            "vlm_response",
            {}
        )
        .get(
            "stop_reason"
        )
    )

    if str(
        stop_reason
        or ""
    ).lower() in {
        "length",
        "max_tokens",
        "model_context_window_exceeded",
    }:
        truncated_ids.add(
            question_id
        )

unresolved_ids = {
    str(
        record.get(
            "question_id",
            "",
        )
    )
    for record in errors
    if record.get(
        "question_id"
    )
}

all_problem_ids = sorted(
    invalid_ids
    | truncated_ids
    | unresolved_ids,
    key=natural_sort,
)

print("Required a retry but succeeded:")
print(
    sorted(
        retried_ids
        - invalid_ids
        - truncated_ids
        - unresolved_ids,
        key=natural_sort,
    )
)

print("\nInvalid or empty:")
print(
    sorted(
        invalid_ids,
        key=natural_sort,
    )
)

print("\nTruncated:")
print(
    sorted(
        truncated_ids,
        key=natural_sort,
    )
)

print("\nUnresolved errors:")
print(
    sorted(
        unresolved_ids,
        key=natural_sort,
    )
)

print("\nQuestions requiring selective retry:")
print(all_problem_ids)


In [ ]:
import json

# ---------------------------------------------------------------------------
# Selective retry
# ---------------------------------------------------------------------------

RUN_SELECTIVE_RETRY = True

# Pass the model alias directly.
RETRY_MODELS = [
    "balanced_gemini",
]

RETRY_QUESTION_IDS = [
    "M026",
    "T027",
    "T038",
]

# Always use a new run ID for a new retry batch.
RETRY_RUN_ID = "retry_balanced_gemini_01"

if RUN_SELECTIVE_RETRY:
    if not RETRY_MODELS:
        raise ValueError(
            "RETRY_MODELS is empty. Add at least one model alias."
        )

    if not RETRY_QUESTION_IDS:
        raise ValueError(
            "RETRY_QUESTION_IDS is empty."
        )

    if not RETRY_RUN_ID:
        raise ValueError(
            "Set a unique RETRY_RUN_ID."
        )

    # Remove accidental duplicate question IDs while preserving order.
    RETRY_QUESTION_IDS = list(
        dict.fromkeys(RETRY_QUESTION_IDS)
    )

    print("Retry model or models:")

    for model_alias in RETRY_MODELS:
        resolved_model = CFG.set_vlm_model(
            model_alias
        )

        print(
            f"  {model_alias} -> {resolved_model}"
        )

    # The selected VLM performs both figure filtering and final answering.
    CFG.use_vlm_figure_filter = True
    CFG.set_filter_model(None)

    print(
        "\nModels passed to runner:",
        RETRY_MODELS,
    )
    print(
        "Questions to retry:",
        len(RETRY_QUESTION_IDS),
    )
    print(
        "Figure-filter override:",
        CFG.vlm_model_filter,
    )
    print(
        "Filter policy:",
        "follow the selected answer model",
    )
    print(
        "Retry run ID:",
        RETRY_RUN_ID,
    )

    retry_paths = run_benchmark(
        CFG=CFG,
        ask_fn=ask,
        questions_path=QUESTIONS_PATH,
        output_root=OUTPUT_ROOT,
        run_id=RETRY_RUN_ID,

        # Pass aliases directly—do not use retry_specs or all_model_specs.
        models=RETRY_MODELS,

        prompt_style=PROMPT_STYLE,
        replicates=1,
        question_ids=RETRY_QUESTION_IDS,
        show_scores=True,
        show_text=True,
        max_attempts=3,
        retry_base_seconds=8.0,
        inter_request_seconds=2.0,
        seed=20260710,
        resume=True,
        rerun_errors=True,
        strict_benchmark_hash=True,
        strict_question_count=True,
        store_serialized_return=True,
        store_raw_debug=True,
        echo_answer_preview=True,
    )

    retry_summary = validate_run_outputs(
        retry_paths["run_dir"],
        RETRY_RUN_ID,
    )

    print(
        json.dumps(
            retry_summary,
            indent=2,
        )
    )

    expected_records = (
        len(RETRY_QUESTION_IDS)
        * len(RETRY_MODELS)
    )

    assert retry_summary["ok_records"] == expected_records, (
        f"Expected {expected_records} successful records, "
        f"but received {retry_summary['ok_records']}."
    )

    assert retry_summary["error_records"] == 0, (
        "The retry contains unresolved error records."
    )

    assert (
        retry_summary["unresolved_error_records"]
        == 0
    ), (
        "The retry contains unresolved provider or runtime failures."
    )

    print(
        "\nSelective retry completed successfully:",
        f"{retry_summary['ok_records']}/{expected_records}",
    )
    print(
        "Run directory:",
        retry_paths["run_dir"],
    )

else:
    print("Selective retry is disabled.")

## 13. Selective retry

Rerun only genuine runtime/provider failures, empty answers, confirmed
truncation, missing records, or corrupted outputs.

Do not rerun an answer only because it is wrong.


In [ ]:
import json

# ---------------------------------------------------------------------------
# Selective retry
# ---------------------------------------------------------------------------

RUN_SELECTIVE_RETRY = True

# Pass the model alias directly.
RETRY_MODELS = [
    "frontier_gemini",
]

RETRY_QUESTION_IDS = [
    "M017",
    "M018",
    "M019",
    "M020",
    "M021",
    "M022",
    "M023",
    "M024",
    "M025",
    "M026",
    "M027",
    "M028",
    "M029",
    "M030",
]

# Always use a new run ID for a new retry batch.
RETRY_RUN_ID = "retry_frontier_gemini_01"

if RUN_SELECTIVE_RETRY:
    if not RETRY_MODELS:
        raise ValueError(
            "RETRY_MODELS is empty. Add at least one model alias."
        )

    if not RETRY_QUESTION_IDS:
        raise ValueError(
            "RETRY_QUESTION_IDS is empty."
        )

    if not RETRY_RUN_ID:
        raise ValueError(
            "Set a unique RETRY_RUN_ID."
        )

    # Remove accidental duplicate question IDs while preserving order.
    RETRY_QUESTION_IDS = list(
        dict.fromkeys(RETRY_QUESTION_IDS)
    )

    print("Retry model or models:")

    for model_alias in RETRY_MODELS:
        resolved_model = CFG.set_vlm_model(
            model_alias
        )

        print(
            f"  {model_alias} -> {resolved_model}"
        )

    # The selected VLM performs both figure filtering and final answering.
    CFG.use_vlm_figure_filter = True
    CFG.set_filter_model(None)

    print(
        "\nModels passed to runner:",
        RETRY_MODELS,
    )
    print(
        "Questions to retry:",
        len(RETRY_QUESTION_IDS),
    )
    print(
        "Figure-filter override:",
        CFG.vlm_model_filter,
    )
    print(
        "Filter policy:",
        "follow the selected answer model",
    )
    print(
        "Retry run ID:",
        RETRY_RUN_ID,
    )

    retry_paths = run_benchmark(
        CFG=CFG,
        ask_fn=ask,
        questions_path=QUESTIONS_PATH,
        output_root=OUTPUT_ROOT,
        run_id=RETRY_RUN_ID,

        # Pass aliases directly—do not use retry_specs or all_model_specs.
        models=RETRY_MODELS,

        prompt_style=PROMPT_STYLE,
        replicates=1,
        question_ids=RETRY_QUESTION_IDS,
        show_scores=True,
        show_text=True,
        max_attempts=3,
        retry_base_seconds=8.0,
        inter_request_seconds=2.0,
        seed=20260710,
        resume=True,
        rerun_errors=True,
        strict_benchmark_hash=True,
        strict_question_count=True,
        store_serialized_return=True,
        store_raw_debug=True,
        echo_answer_preview=True,
    )

    retry_summary = validate_run_outputs(
        retry_paths["run_dir"],
        RETRY_RUN_ID,
    )

    print(
        json.dumps(
            retry_summary,
            indent=2,
        )
    )

    expected_records = (
        len(RETRY_QUESTION_IDS)
        * len(RETRY_MODELS)
    )

    assert retry_summary["ok_records"] == expected_records, (
        f"Expected {expected_records} successful records, "
        f"but received {retry_summary['ok_records']}."
    )

    assert retry_summary["error_records"] == 0, (
        "The retry contains unresolved error records."
    )

    assert (
        retry_summary["unresolved_error_records"]
        == 0
    ), (
        "The retry contains unresolved provider or runtime failures."
    )

    print(
        "\nSelective retry completed successfully:",
        f"{retry_summary['ok_records']}/{expected_records}",
    )
    print(
        "Run directory:",
        retry_paths["run_dir"],
    )

else:
    print("Selective retry is disabled.")

## 14. Validate and package a run

Set `RUN_ID_TO_PACKAGE` to a completed smoke, primary, or retry run.


In [ ]:
import json
import shutil
from pathlib import Path

RUN_ID_TO_PACKAGE = PRIMARY_RUN_ID

run_dir_to_package = (
    OUTPUT_ROOT
    / RUN_ID_TO_PACKAGE
)

if run_dir_to_package.exists():
    validation = validate_run_outputs(
        run_dir_to_package,
        RUN_ID_TO_PACKAGE,
    )

    print(
        json.dumps(
            validation,
            indent=2,
        )
    )

    archive_base = (
        Path(CFG.base_dir)
        / f"{RUN_ID_TO_PACKAGE}_outputs"
    )

    archive_path = shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=run_dir_to_package,
    )

    print("Archive:", archive_path)

    # Optional direct Colab download:
    #
    # from google.colab import files
    # files.download(archive_path)

else:
    print(
        "Run directory does not exist:",
        run_dir_to_package,
    )


## GitHub checklist

Before uploading this notebook:

- Keep `RUN_FULL_BENCHMARK=False`.
- Keep `RUN_SELECTIVE_RETRY=False`.
- Remove all cell outputs.
- Do not include API keys.
- Do not include benchmark outputs, gold answers, evaluator files,
  Qdrant data, page images, or model caches.
- Keep `CFG.set_filter_model(None)` unchanged so the selected VLM
  performs figure filtering and answering.
